# Missão Aurora Siger — Ignition Zero Core

## Sistema de Verificação de Voo Embarcado

> **Contexto:** Na Nova Corrida Espacial, o atraso de comunicação Terra–Marte pode chegar a **22 minutos**. O sistema embarcado deve ser **determinístico e autônomo** durante a janela crítica de decolagem.

---

### Pilares do Sistema

| Pilar | Papel no Sistema |
|---|---|
| Ciência da Computação | Digitalização da realidade física via Thresholding |
| Automação com Python | Lógica de segurança modular e determinística |
| Inteligência Artificial | Análise probabilística e detecção de padrões |
| Energia e Sustentabilidade | Otimização de ciclos de instrução e footprint mínimo |


## 1. Matriz de Variáveis Críticas

| Variável | Descrição Técnica | Tipo | Importância |
|---|---|---|---|
| `temperatura` | Monitoramento térmico de sistemas críticos | `float` (°C) | Estabilidade térmica |
| `integridade` | Extensômetros e sensores ultrassônicos | `bool` (0/1) | Elasticidade do casco |
| `energia` | State of Charge (SoC) das baterias | `%` | Autonomia de suporte à vida |
| `pressao` | Sensores piezoresistivos nos tanques | `float` (PSI) | Previne falhas catastróficas |
| `modulos` | Diagnóstico de prontidão dos módulos | `bool` (flag) | Sistemas secundários ativos |


## 2. Pseudocódigo Estruturado — Arquitetura Fail-Safe

> Filosofia **AND Gate**: a decisão "PRONTO PARA DECOLAR" exige que **todas** as condições sejam verdadeiras simultaneamente. Qualquer anomalia dispara o protocolo de abortagem.

```
INÍCIO
    // Validação de entrada e leitura de sensores
    LER telemetria_bruta
    VALIDAR presença e tipo de todos os sensores obrigatórios
    SE dados inválidos → ABORTAR com log de erro

    // Definição de faixas nominais (Safe Zones)
    CONSTANTE LIMITE_TEMP_MAX  = 35.0   // °C
    CONSTANTE ENERGIA_MINIMA   = 80.0   // %
    CONSTANTE PRESSAO_MIN      = 90.0   // PSI
    CONSTANTE PRESSAO_MAX      = 110.0  // PSI

    // Thresholding: digitalização de sinais analógicos
    temp_ok  ← telemetria.temperatura <= LIMITE_TEMP_MAX
    integ_ok ← telemetria.integridade == 1
    ener_ok  ← telemetria.energia >= ENERGIA_MINIMA
    press_ok ← PRESSAO_MIN <= telemetria.pressao <= PRESSAO_MAX
    mod_ok   ← telemetria.modulos == 1

    // Lógica Gate Strategy (AND Gate — Fail-Safe)
    SE temp_ok E integ_ok E ener_ok E press_ok E mod_ok ENTÃO

        GERAR_LOG "STATUS: PRONTO PARA DECOLAR - SISTEMAS NOMINAIS"
        CALCULAR autonomia_energetica(telemetria.energia)
        INICIAR SEQUÊNCIA_IGNICAO

    SENÃO

        GERAR_LOG "STATUS: DECOLAGEM ABORTADA - FALHA DE SEGURANÇA DETECTADA"
        IDENTIFICAR_ANOMALIA(telemetria, estados)   // caixa-preta auditável
        BLOQUEAR_IGNICAO

    FIM SE
FIM
```

| Variável lógica | Condição de aprovação |
|---|---|
| `temp_ok` | temperatura ≤ 35.0 °C |
| `integ_ok` | integridade == 1 (estrutura nominal) |
| `ener_ok` | energia ≥ 80% (SoC mínimo) |
| `press_ok` | 90.0 ≤ pressão ≤ 110.0 PSI |
| `mod_ok` | módulos == 1 (todos ativos) |

## 2. Importações e Configuração

In [ ]:
from typing import Dict, List, Tuple
from dataclasses import dataclass, field
from datetime import datetime, timezone

# Constantes de Safe Zones (Limiarização)
LIMITE_TEMP_MAX: float = 35.0
ENERGIA_MINIMA: float = 80.0
PRESSAO_MIN: float = 90.0
PRESSAO_MAX: float = 110.0

# Constantes energéticas
CAPACIDADE_TOTAL_KWH: float = 500.0
CONSUMO_DECOLAGEM_KWH: float = 120.0
PERDAS_ENERGETICAS: float = 0.08

print("Configuração carregada com sucesso.")
print(f"  Temp máx.: {LIMITE_TEMP_MAX}°C | Energia mín.: {ENERGIA_MINIMA}% | Pressão: {PRESSAO_MIN}–{PRESSAO_MAX} PSI")

## 3. Estruturas de Dados

In [ ]:
@dataclass
class Telemetria:
    temperatura: float
    integridade: float
    energia: float
    pressao: float
    modulos: float
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

@dataclass
class ResultadoVerificacao:
    pronto: bool
    status: str
    anomalias: List[str]
    timestamp: str
    telemetria: Telemetria
    autonomia_horas: float = 0.0

print("Estruturas de dados definidas.")

## 4. Módulos do Sistema

### 4.1 Validação de Entrada

In [ ]:
SENSORES_OBRIGATORIOS = ["temperatura", "integridade", "energia", "pressao", "modulos"]

def validar_entrada(telemetria: Dict[str, float]) -> Tuple[bool, List[str]]:
    """Valida presença e tipo de todos os campos obrigatórios."""
    erros: List[str] = []
    for sensor in SENSORES_OBRIGATORIOS:
        if sensor not in telemetria:
            erros.append(f"Sensor ausente: '{sensor}'")
        elif not isinstance(telemetria[sensor], (int, float)):
            erros.append(f"Tipo inválido para '{sensor}'")
    return len(erros) == 0, erros

# Teste rápido
ok, errs = validar_entrada({"temperatura": 28.5, "integridade": 1.0,
                             "energia": 95.0, "pressao": 102.3, "modulos": 1.0})
print(f"Entrada válida: {ok} | Erros: {errs}")

ok2, errs2 = validar_entrada({"temperatura": 28.5})
print(f"Entrada inválida: {ok2} | Erros: {errs2}")

### 4.2 Thresholding — Digitalização de Sinais

In [ ]:
def digitalizar_sinais(telemetria: Dict[str, float]) -> Dict[str, bool]:
    """Converte grandezas físicas em estados lógicos booleanos."""
    return {
        "temp_ok":  telemetria["temperatura"] <= LIMITE_TEMP_MAX,
        "integ_ok": telemetria["integridade"] == 1.0,
        "ener_ok":  telemetria["energia"] >= ENERGIA_MINIMA,
        "press_ok": PRESSAO_MIN <= telemetria["pressao"] <= PRESSAO_MAX,
        "mod_ok":   telemetria["modulos"] == 1.0,
    }

exemplo = {"temperatura": 28.5, "integridade": 1.0,
           "energia": 95.0, "pressao": 102.3, "modulos": 1.0}
estados = digitalizar_sinais(exemplo)
for k, v in estados.items():
    print(f"  {k}: {'✓ NOMINAL' if v else '✗ FALHA'}")

### 4.3 Identificação de Anomalias

In [ ]:
def identificar_anomalias(telemetria: Dict[str, float], estados: Dict[str, bool]) -> List[str]:
    """Gera log auditável de anomalias (caixa-preta / black-box)."""
    anomalias: List[str] = []
    if not estados["temp_ok"]:
        anomalias.append(f"TEMPERATURA CRÍTICA: {telemetria['temperatura']}°C (limite: {LIMITE_TEMP_MAX}°C)")
    if not estados["integ_ok"]:
        anomalias.append(f"FALHA DE INTEGRIDADE ESTRUTURAL: flag={telemetria['integridade']}")
    if not estados["ener_ok"]:
        anomalias.append(f"ENERGIA INSUFICIENTE: {telemetria['energia']}% (mínimo: {ENERGIA_MINIMA}%)")
    if not estados["press_ok"]:
        anomalias.append(f"PRESSÃO FORA DO NOMINAL: {telemetria['pressao']} PSI (faixa: {PRESSAO_MIN}–{PRESSAO_MAX} PSI)")
    if not estados["mod_ok"]:
        anomalias.append(f"MÓDULOS INATIVOS: flag={telemetria['modulos']}")
    return anomalias

print("Módulo de identificação de anomalias carregado.")

### 4.4 Análise Energética

**Fórmula de Autonomia:**

$$\text{Autonomia} = \frac{(\text{Capacidade Total} \times \text{Carga Atual}) - \text{Consumo Decolagem}}{\text{Perdas Energéticas}}$$

In [ ]:
def calcular_autonomia(carga_atual_pct: float) -> float:
    """
    Autonomia = ((Capacidade_Total × Carga_Atual) - Consumo_Decolagem)
                / Perdas_Energéticas
    """
    energia_disponivel = CAPACIDADE_TOTAL_KWH * (carga_atual_pct / 100.0)
    energia_util = energia_disponivel - CONSUMO_DECOLAGEM_KWH
    if energia_util <= 0:
        return 0.0
    return round(energia_util / (CAPACIDADE_TOTAL_KWH * PERDAS_ENERGETICAS), 2)

# Demonstração com diferentes SoC
cargas = [60, 70, 80, 90, 95, 100]
print(f"{'SoC (%)':>10} | {'Energia Disponível (kWh)':>25} | {'Autonomia (horas)':>18}")
print("-" * 60)
for c in cargas:
    disp = CAPACIDADE_TOTAL_KWH * (c / 100.0)
    aut = calcular_autonomia(c)
    print(f"{c:>10} | {disp:>25.1f} | {aut:>18.2f}")

### 4.5 Lógica de Decisão Principal — AND Gate (Fail-Safe)

In [ ]:
def validar_sistemas(telemetria_raw: Dict[str, float]) -> ResultadoVerificacao:
    """Verificação determinística completa com filosofia Fail-Safe."""
    timestamp = datetime.now(timezone.utc).isoformat()

    entrada_valida, erros_entrada = validar_entrada(telemetria_raw)
    if not entrada_valida:
        dados = Telemetria(**{k: 0.0 for k in SENSORES_OBRIGATORIOS})
        return ResultadoVerificacao(
            pronto=False, status="ERRO: DADOS INVÁLIDOS",
            anomalias=erros_entrada, timestamp=timestamp,
            telemetria=dados, autonomia_horas=0.0
        )

    dados = Telemetria(
        temperatura=telemetria_raw["temperatura"],
        integridade=telemetria_raw["integridade"],
        energia=telemetria_raw["energia"],
        pressao=telemetria_raw["pressao"],
        modulos=telemetria_raw["modulos"],
        timestamp=timestamp,
    )
    estados = digitalizar_sinais(telemetria_raw)
    anomalias = identificar_anomalias(telemetria_raw, estados)
    pronto = all(estados.values())
    autonomia = calcular_autonomia(telemetria_raw["energia"]) if pronto else 0.0
    status = (
        "STATUS: PRONTO PARA DECOLAR - SISTEMAS NOMINAIS"
        if pronto else
        "STATUS: DECOLAGEM ABORTADA - FALHA DE SEGURANÇA DETECTADA"
    )
    return ResultadoVerificacao(
        pronto=pronto, status=status, anomalias=anomalias,
        timestamp=timestamp, telemetria=dados, autonomia_horas=autonomia
    )

print("Módulo de decisão carregado.")

## 5. Simulações de Cenários

### Cenário 1: Telemetria Nominal (Decolagem Autorizada)

In [ ]:
telemetria_nominal = {
    "temperatura": 28.5,
    "integridade": 1.0,
    "energia": 95.0,
    "pressao": 102.3,
    "modulos": 1.0,
}

r = validar_sistemas(telemetria_nominal)
print(f"Resultado: {r.status}")
print(f"Pronto   : {r.pronto}")
print(f"Autonomia: {r.autonomia_horas} horas")
print(f"Anomalias: {r.anomalias}")

### Cenário 2: Temperatura Crítica

In [ ]:
telemetria_temp_alta = {
    "temperatura": 42.1,   # ACIMA DO LIMITE
    "integridade": 1.0,
    "energia": 92.0,
    "pressao": 100.0,
    "modulos": 1.0,
}

r = validar_sistemas(telemetria_temp_alta)
print(f"Resultado: {r.status}")
for a in r.anomalias:
    print(f"  >> {a}")

### Cenário 3: Múltiplas Falhas Simultâneas

In [ ]:
telemetria_multiplas_falhas = {
    "temperatura": 42.1,   # FALHA: acima do limite
    "integridade": 1.0,
    "energia": 67.5,       # FALHA: abaixo do mínimo
    "pressao": 85.0,       # FALHA: abaixo da faixa
    "modulos": 0.0,        # FALHA: módulos inativos
}

r = validar_sistemas(telemetria_multiplas_falhas)
print(f"Resultado: {r.status}")
print(f"Total de anomalias: {len(r.anomalias)}")
for i, a in enumerate(r.anomalias, 1):
    print(f"  [{i}] {a}")

### Cenário 4: Pressão de Propelente Fora dos Limites

In [ ]:
telemetria_sobrepressao = {
    "temperatura": 30.0,
    "integridade": 1.0,
    "energia": 88.0,
    "pressao": 125.7,      # FALHA: sobrepressão
    "modulos": 1.0,
}

r = validar_sistemas(telemetria_sobrepressao)
print(f"Resultado: {r.status}")
for a in r.anomalias:
    print(f"  >> {a}")

### Cenário 5: Dados Corrompidos (Sensor Ausente)

In [ ]:
telemetria_corrompida = {
    "temperatura": 28.5,
    # 'integridade' ausente — simula falha de sensor/transmissão
    "energia": 91.0,
    "pressao": 101.0,
    "modulos": 1.0,
}

r = validar_sistemas(telemetria_corrompida)
print(f"Resultado: {r.status}")
for a in r.anomalias:
    print(f"  >> {a}")

## 6. Análise Energética Detalhada

In [ ]:
carga = 95.0
energia_disponivel = CAPACIDADE_TOTAL_KWH * (carga / 100.0)
energia_util = energia_disponivel - CONSUMO_DECOLAGEM_KWH
perdas_absolutas = energia_disponivel * PERDAS_ENERGETICAS
autonomia = calcular_autonomia(carga)

print("ANÁLISE ENERGÉTICA — IGNITION ZERO")
print("=" * 45)
print(f"Capacidade Total         : {CAPACIDADE_TOTAL_KWH:.1f} kWh")
print(f"Carga Atual (SoC)        : {carga:.1f}%")
print(f"Energia Disponível       : {energia_disponivel:.1f} kWh")
print(f"Consumo na Decolagem     : {CONSUMO_DECOLAGEM_KWH:.1f} kWh")
print(f"Energia Útil Remanescente: {energia_util:.1f} kWh")
print(f"Perdas Térmicas/Conversão: {perdas_absolutas:.1f} kWh ({PERDAS_ENERGETICAS*100:.0f}%)")
print(f"Autonomia Estimada       : {autonomia:.2f} horas")

## 7. Análise Assistida por IA

A IA atua como **assistente estratégica** — uma camada analítica **probabilística**, complementar à lógica **determinística** do script Python. Ela **não** autoriza decolagens; ela identifica tendências, classifica riscos e sugere mitigações que algoritmos de regra fixa podem ignorar.

Conforme orientado na missão, os dados de telemetria foram submetidos a uma IA (Claude/Anthropic) com três solicitações:

1. **Classificação dos dados** de telemetria em categorias de risco
2. **Identificação de possíveis anomalias** nos padrões dos sensores
3. **Sugestões de risco** e mitigação

---

### 7.1 Prompt 1 — Classificação dos Dados

**Prompt enviado à IA:**

> "Classifique os seguintes dados de telemetria da nave Aurora Siger em categorias de risco aeroespacial (Nominal, Degradado, Crítico). Justifique cada classificação com base nas faixas seguras definidas: temperatura ≤ 35°C, integridade == 1, energia ≥ 80%, pressão entre 90–110 PSI, módulos == 1."
>
> Cenário A: temperatura=28.5, integridade=1, energia=95, pressão=102.3, módulos=1
>
> Cenário B: temperatura=42.1, integridade=1, energia=67.5, pressão=85.0, módulos=1

**Resposta da IA:**

> **Cenário A — Classificação: NOMINAL**
> Todos os parâmetros estão dentro das faixas seguras predefinidas. A temperatura de 28.5°C possui margem de 6.5°C em relação ao limite máximo de 35°C. O SoC de 95% excede significativamente o mínimo de 80%, proporcionando reserva energética confortável. A pressão de 102.3 PSI está centralizada na faixa nominal (90–110 PSI), indicando estabilidade nos tanques de propelente. Integridade estrutural e módulos de controle estão ativos. **Recomendação: sistemas aptos para decolagem.**
>
> **Cenário B — Classificação: CRÍTICO**
> Três dos cinco parâmetros estão fora das faixas seguras:
> - **Temperatura (42.1°C):** Excede o limite de 35°C em 7.1°C — risco de degradação térmica em componentes eletrônicos e possível expansão descontrolada de propelente.
> - **Energia (67.5%):** 12.5 pontos percentuais abaixo do mínimo de 80% — insuficiente para garantir autonomia dos sistemas de suporte à vida e navegação durante a fase de ascensão.
> - **Pressão (85.0 PSI):** 5 PSI abaixo do limite inferior de 90 PSI — indica possível vazamento nos tanques ou falha de pressurização, com risco de subpressão durante a ignição.
> **Recomendação: decolagem deve ser abortada. Priorizar diagnóstico térmico e inspeção de integridade dos tanques.**

### 7.2 Prompt 2 — Identificação de Possíveis Anomalias

**Prompt enviado à IA:**

> "Com base nos dados de telemetria do Cenário B (temperatura=42.1°C, integridade=1, energia=67.5%, pressão=85.0 PSI, módulos=1), identifique possíveis anomalias operacionais e correlações entre os parâmetros que possam indicar falhas sistêmicas não evidentes em uma análise individual."

**Resposta da IA:**

> A análise cruzada dos parâmetros do Cenário B revela correlações preocupantes:
>
> **1. Correlação Temperatura × Pressão:**
> A temperatura elevada de 42.1°C combinada com a subpressão de 85.0 PSI nos tanques sugere uma possível anomalia no sistema de pressurização. O calor excessivo pode estar causando vaporização parcial do propelente, o que paradoxalmente reduz a pressão líquida no tanque ao invés de aumentá-la — indicativo de um possível vazamento com perda de massa no propelente criogênico.
>
> **2. Correlação Energia × Temperatura:**
> O SoC de 67.5% em combinação com a temperatura de 42.1°C pode indicar que os sistemas de refrigeração estão operando em regime de emergência, consumindo energia acima do previsto para tentar estabilizar a temperatura. Isso explicaria a drenagem prematura das baterias.
>
> **3. Anomalia Latente — Integridade:**
> Embora a integridade estrutural marque 1 (nominal), os extensômetros podem não capturar micro-fadigas causadas por ciclos térmicos extremos. A combinação de estresse térmico prolongado com variação de pressão nos tanques pode gerar tensões mecânicas que só se manifestariam durante a vibração da decolagem.
>
> **Conclusão:** Os dados não representam três falhas independentes, mas possivelmente uma **cadeia causal única** — um vazamento de propelente gerando perda de pressão, aquecimento compensatório e consumo energético elevado. Recomenda-se inspeção física dos tanques antes de qualquer tentativa de reabastecimento ou repressurização.

---

### 7.3 Prompt 3 — Sugestões de Risco e Mitigação

**Prompt enviado à IA:**

> "Considerando os dados de telemetria com falhas identificadas (temperatura=42.1°C, energia=67.5%, pressão=85.0 PSI), sugira protocolos de mitigação de risco para cada anomalia, priorizando a segurança da tripulação e a preservação dos sistemas da nave Aurora Siger."

**Resposta da IA:**

> **Protocolo de Mitigação — Prioridade por Criticidade:**
>
> **PRIORIDADE 1 — Pressão dos Tanques (85.0 PSI)**
> - Ação imediata: isolar válvulas de propelente e executar teste de estanqueidade nos tanques
> - Verificar sensores piezoresistivos para descartar leitura falsa (redundância de sensores)
> - Se confirmado vazamento: acionar protocolo de purga controlada e reabastecimento
> - Critério de revalidação: pressão estável entre 90–110 PSI por no mínimo 15 minutos contínuos
>
> **PRIORIDADE 2 — Temperatura (42.1°C)**
> - Ativar sistema de refrigeração auxiliar (backup cooling loop)
> - Redirecionar dissipação térmica para radiadores secundários
> - Monitorar taxa de resfriamento: a temperatura deve cair ≥ 1°C/min para confirmar eficácia
> - Se a temperatura não recuar para ≤ 35°C em 20 minutos: avaliar adiamento da janela de lançamento
>
> **PRIORIDADE 3 — Energia (67.5%)**
> - Desativar módulos não essenciais (científicos, comunicação de longo alcance) para reduzir consumo
> - Acionar carregamento de emergência via painéis solares ou fonte externa na plataforma
> - Critério mínimo: SoC ≥ 80% antes de qualquer nova tentativa de verificação pré-decolagem
> - Recalcular autonomia pós-decolagem com nova carga antes de autorizar ignição
>
> **Protocolo Geral:**
> Após todas as correções, executar novamente o ciclo completo de verificação (AND Gate). A decolagem só será reautorizada quando **todos os cinco parâmetros** retornarem ao estado nominal simultaneamente.

## 8. Execução Completa — Relatório de Log

In [ ]:
def imprimir_relatorio(resultado: ResultadoVerificacao) -> None:
    sep = "=" * 55
    print(sep)
    print("  AURORA SIGER — IGNITION ZERO CORE")
    print(f"  Timestamp UTC: {resultado.timestamp}")
    print(sep)
    print(f"\n  [RESULTADO]: {resultado.status}\n")
    t = resultado.telemetria
    print("  SENSORES:")
    print(f"    Temperatura : {t.temperatura}°C")
    print(f"    Integridade : {int(t.integridade)}")
    print(f"    Energia     : {t.energia}%")
    print(f"    Pressão     : {t.pressao} PSI")
    print(f"    Módulos     : {int(t.modulos)}")
    if resultado.pronto:
        print(f"\n  Autonomia estimada: {resultado.autonomia_horas} horas")
        print("  >> Iniciando Sequência de Ignição...")
    else:
        print(f"\n  ANOMALIAS ({len(resultado.anomalias)}):")
        for i, a in enumerate(resultado.anomalias, 1):
            print(f"    [{i}] {a}")
        print("  >> Ignição Bloqueada.")
    print(sep)

print("--- CENÁRIO NOMINAL ---")
imprimir_relatorio(validar_sistemas({"temperatura": 28.5, "integridade": 1.0,
                                      "energia": 95.0, "pressao": 102.3, "modulos": 1.0}))

print("\n--- CENÁRIO COM FALHAS ---")
imprimir_relatorio(validar_sistemas({"temperatura": 42.1, "integridade": 1.0,
                                      "energia": 67.5, "pressao": 85.0, "modulos": 1.0}))

---

## 9. Reflexão Crítica

### 9.1 Ética e Responsabilidade

A engenharia de sistemas embarcados para missões espaciais carrega uma responsabilidade ética inegociável: decisões algorítmicas afetam diretamente vidas humanas. Quando projetamos um sistema que decide autonomamente se uma nave está apta para decolar, estamos delegando a uma máquina uma decisão que, se errada, pode ser fatal. Por isso, a filosofia Fail-Safe adotada neste projeto não é apenas uma escolha técnica — é um imperativo ético. A opção por abortar diante de qualquer anomalia reflete o princípio de que, na dúvida, a segurança da tripulação deve prevalecer sobre cronogramas e custos de missão.

Além disso, a transparência algorítmica é uma exigência moral. O sistema foi projetado para gerar logs auditáveis que identificam com precisão cada anomalia detectada, incluindo o valor medido e o limite esperado. Se a nave abortar uma decolagem, qualquer engenheiro ou comitê de revisão deve ser capaz de rastrear exatamente o que aconteceu e por quê. Essa rastreabilidade é o equivalente computacional da caixa-preta de uma aeronave — sem ela, não há como aprender com falhas nem responsabilizar decisões.

### 9.2 Impacto Social da Exploração Espacial

A exploração espacial historicamente funcionou como catalisadora de avanços tecnológicos que beneficiam a sociedade como um todo. Tecnologias desenvolvidas para missões da NASA, como filtros de purificação de água, sensores médicos portáteis e materiais de isolamento térmico, hoje são amplamente utilizadas no cotidiano. A Missão Aurora Siger, embora simulada, reflete esse ciclo: os conceitos de automação, análise de dados e eficiência energética aqui aplicados são diretamente transferíveis para setores como saúde, transporte, agricultura e infraestrutura.

Contudo, é fundamental questionar quem se beneficia dessa expansão tecnológica. A exploração espacial não deve servir apenas a nações e corporações com recursos para financiá-la. A democratização do conhecimento espacial — incluindo a publicação aberta de código, dados e metodologias — é essencial para garantir que os avanços alcancem comunidades que mais precisam de soluções tecnológicas. Disponibilizar este projeto em repositório público é um gesto pequeno, mas alinhado com esse princípio.

### 9.3 Sustentabilidade Tecnológica

No contexto espacial, sustentabilidade não é um conceito abstrato — é uma questão de sobrevivência. Cada ciclo de CPU gera calor e consome energia de baterias que não podem ser facilmente recarregadas a milhões de quilômetros da Terra. Otimizar algoritmos significa, literalmente, estender a vida útil da nave e da tripulação.

Este projeto foi desenvolvido com zero dependências externas, utilizando exclusivamente a biblioteca padrão do Python. Essa decisão não é apenas pragmática — ela reduz o footprint computacional, minimiza o consumo de memória e garante que o código funcione em ambientes restritos sem necessidade de instalação de pacotes adicionais. O conceito de Code Longevity (longevidade do código) também foi priorizado: type hints, dataclasses e funções puras sem efeitos colaterais garantem que o sistema seja mantido e evoluído por outras equipes no futuro, sem acumular dívida técnica.

Na Terra, esses mesmos princípios se traduzem em responsabilidade ambiental. Data centers consomem quantidades massivas de energia, e software ineficiente amplifica esse consumo. Escrever código limpo e otimizado é uma forma concreta de reduzir o impacto ambiental da tecnologia — uma responsabilidade que todo engenheiro de software deveria abraçar, esteja ele programando para Marte ou para o servidor ao lado.

---

*Missão Aurora Siger — Ignition Zero Core — Desenvolvido para a disciplina de Automação com Python e Fundamentos de IA — FIAP*